In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("star-to-clickhouse")
    .config(
        "spark.jars.packages",
        "org.postgresql:postgresql:42.5.4,com.clickhouse:clickhouse-jdbc:0.4.6",
    )
    .getOrCreate()
)

PG_URL = "jdbc:postgresql://postgres:5432/bigdata_spark"
PG_PROPS = {"user": "postgres", "password": "postgres", "driver": "org.postgresql.Driver"}

CH_URL = "jdbc:ch://clickhouse:8123/default"
CH_PROPS = {
    "user": "user",
    "password": "password",
    "driver": "com.clickhouse.jdbc.ClickHouseDriver",
    "compress": "0",
    "decompress": "0",
}

def pg_read(table: str):
    return spark.read.jdbc(url=PG_URL, table=table, properties=PG_PROPS)

def ch_write(df, table: str):
    df.write.jdbc(url=CH_URL, table=table, mode="overwrite", properties=CH_PROPS)

TABLES = ["fact_sales", "dim_product", "dim_customer", "dim_seller",
          "dim_store", "dim_supplier"]

for t in TABLES:
    pg_read(t).createOrReplaceTempView(t)

print("Все таблицы загружены")

Все таблицы загружены


## Витрина продаж по продуктам

In [2]:
rep_products = spark.sql("""
    SELECT
        p.name             AS product_name,
        p.category_name,
        p.brand_name,
        p.rating,
        SUM(f.quantity)    AS total_qty,
        SUM(f.total_price) AS total_revenue,
        COUNT(f.sale_id)   AS num_transactions
    FROM fact_sales f
    JOIN dim_product p ON f.product_id = p.product_id
    GROUP BY p.name, p.category_name, p.brand_name, p.rating
    ORDER BY total_revenue DESC
    LIMIT 10
""")

ch_write(rep_products, "report_product_sales")
print("report_product_sales - готово")

report_product_sales - готово


## Витрина продаж по клиентам

In [3]:
rep_customers = spark.sql("""
    SELECT
        c.customer_id,
        c.first_name,
        c.last_name,
        c.country_name,
        SUM(f.total_price)  AS total_spent,
        AVG(f.total_price)  AS avg_check,
        COUNT(f.sale_id)    AS num_orders
    FROM fact_sales f
    JOIN dim_customer c ON f.customer_id = c.customer_id
    GROUP BY c.customer_id, c.first_name, c.last_name, c.country_name
    ORDER BY total_spent DESC
    LIMIT 10
""")

ch_write(rep_customers, "report_customer_sales")
print("report_customer_sales - готово")

report_customer_sales - готово


## Витрина продаж по времени

In [4]:
rep_time = spark.sql("""
    WITH monthly AS (
        SELECT
            year(f.sale_date)        AS sale_year,
            month(f.sale_date)       AS sale_month,
            SUM(f.total_price)       AS monthly_revenue,
            AVG(f.total_price)       AS avg_order_size,
            SUM(f.quantity)          AS items_sold,
            COUNT(DISTINCT f.customer_id) AS unique_customers
        FROM fact_sales f
        GROUP BY year(f.sale_date), month(f.sale_date)
    )
    SELECT
        sale_year,
        sale_month,
        monthly_revenue,
        COALESCE(LAG(monthly_revenue) OVER (ORDER BY sale_year, sale_month), 0) AS prev_month_revenue,
        COALESCE(
            monthly_revenue - LAG(monthly_revenue) OVER (ORDER BY sale_year, sale_month), 0
        ) AS revenue_delta,
        avg_order_size,
        items_sold,
        unique_customers
    FROM monthly
    ORDER BY sale_year, sale_month
""")

ch_write(rep_time, "report_time_sales")
print("report_time_sales - готово")

report_time_sales - готово


## Витрина продаж по магазинам

In [5]:
rep_stores = spark.sql("""
    SELECT
        s.store_name,
        s.city,
        s.country_name,
        SUM(f.total_price)   AS store_revenue,
        AVG(f.total_price)   AS avg_check,
        COUNT(f.sale_id)     AS num_sales,
        SUM(f.quantity)      AS units_sold
    FROM fact_sales f
    JOIN dim_store s ON f.store_id = s.store_id
    GROUP BY s.store_name, s.city, s.country_name
    ORDER BY store_revenue DESC
    LIMIT 5
""")

ch_write(rep_stores, "report_store_sales")
print("report_store_sales - готово")

report_store_sales - готово


## Витрина продаж по поставщикам

In [6]:
rep_suppliers = spark.sql("""
    SELECT
        sup.name             AS supplier_name,
        sup.country_name     AS supplier_country,
        sup.city             AS supplier_city,
        SUM(f.total_price)   AS supplier_revenue,
        AVG(p.price)         AS avg_product_price,
        COUNT(DISTINCT p.product_id) AS num_products
    FROM fact_sales f
    JOIN dim_product  p   ON f.product_id  = p.product_id
    JOIN dim_supplier sup ON f.supplier_id = sup.supplier_id
    GROUP BY sup.name, sup.country_name, sup.city
    ORDER BY supplier_revenue DESC
    LIMIT 5
""")

ch_write(rep_suppliers, "report_supplier_sales")
print("report_supplier_sales - готово")

report_supplier_sales - готово


## Витрина качества продукции

In [7]:
rep_quality = spark.sql("""
    SELECT
        p.name          AS product_name,
        p.brand_name,
        p.rating,
        p.reviews,
        SUM(f.quantity) AS sales_volume,
        SUM(f.total_price) AS sales_revenue
    FROM fact_sales f
    JOIN dim_product p ON f.product_id = p.product_id
    GROUP BY p.name, p.brand_name, p.rating, p.reviews
    ORDER BY p.rating DESC, sales_volume DESC
""")

ch_write(rep_quality, "report_quality")
print("report_quality - готово")

report_quality - готово
